# SLA Risk Scoring for Open Tickets
GitHub Support Operations Analytics - Predictive Risk Model

This notebook calculates real-time risk scores for open tickets using:
- Ticket age (hours since creation)
- Comment activity (engagement level)
- Label weights (security/critical bugs = higher risk)
- Statistical SLA breach probability model

Output: Risk scores stored in Risk_Scores table for real-time dashboard alerts

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")

In [ ]:
# 1. Database Connection
engine = create_engine('mssql+pyodbc://YourServer/YourDB?driver=ODBC+Driver+17+for+SQL+Server')

In [ ]:
# 2. Fetch all OPEN tickets (not closed)
query = """
SELECT 
    f.Ticket_Key,
    r.Repo_Name,
    a.Author_Name,
    f.Created_At,
    f.Label,
    f.Comment_Count,
    f.First_Response_At
FROM Fact_Tickets f
JOIN Dim_Repositories r ON f.Repo_Key = r.Repo_Key
JOIN Dim_Authors a ON f.Author_Key = a.Author_Key
WHERE f.Closed_At IS NULL
  AND f.Created_At >= DATEADD(HOUR, -30, GETDATE())
"""

df_open = pd.read_sql(query, engine)
print(f"Loaded {len(df_open)} open tickets")
print(df_open.head())

In [ ]:
# 3. Calculate Ticket Age (in hours)
df_open['Ticket_Age_Hours'] = (datetime.now() - df_open['Created_At']).dt.total_seconds() / 3600

print("Ticket Age Statistics:")
print(df_open['Ticket_Age_Hours'].describe())

In [ ]:
# 4. Assign Label Weights (Security = High Risk)
label_weights = {
    'security': 1.5,
    'critical-bug': 1.4,
    'vulnerability': 1.5,
    'bug': 1.0,
    'enhancement': 0.3,
    'feature': 0.3,
    'question': 0.5,
    'help wanted': 0.6,
    'documentation': 0.2,
    'other': 0.5
}

df_open['Label_Weight'] = df_open['Label'].str.lower().map(label_weights).fillna(0.5)

print("Label Weight Distribution:")
print(df_open.groupby('Label')['Label_Weight'].first().sort_values(ascending=False))

In [ ]:
# 5. Calculate SLA Breach Probability using Statistical Model
# Model: Probability of breaching 24-hour SLA based on:
#   - Ticket age (hours already spent)
#   - Comment count (engagement/complexity indicator)
#   - Label weight (risk multiplier)

def calculate_breach_probability(row):
    """
    Statistical model for SLA breach probability.
    
    Factors:
    - Time spent: How many hours left to SLA deadline (24h total)
    - Comment activity: More comments = higher complexity = higher risk
    - Label weight: Security/critical bugs have inherent higher risk
    """
    
    ticket_age = row['Ticket_Age_Hours']
    comment_count = row['Comment_Count']
    label_weight = row['Label_Weight']
    
    # SLA threshold: 24 hours
    time_remaining = max(0, 24 - ticket_age)
    
    # Base probability: Time-based
    # If 18 hours passed, 6 hours remaining = 75% breach risk (18/24)
    time_risk = ticket_age / 24
    
    # Comment penalty: Each additional comment increases complexity risk
    # Formula: +5% risk per comment (max 40% bonus)
    comment_risk = min(0.40, comment_count * 0.05)
    
    # Label weight multiplier
    # Security bugs have 1.5x weight, so their breach probability increases 50%
    
    # Combined formula (normalized to 0-100)
    breach_prob = ((time_risk * 0.5) + (comment_risk * 0.3) + (label_weight * 0.2)) * 100
    
    return np.clip(breach_prob, 0, 100)

df_open['SLA_Breach_Probability'] = df_open.apply(calculate_breach_probability, axis=1)

print("\nSLA Breach Probability Distribution:")
print(df_open['SLA_Breach_Probability'].describe())

In [ ]:
# 6. Create Risk Score Classification
def classify_risk(breach_prob):
    """Classify tickets into risk tiers"""
    if breach_prob >= 80:
        return 'CRITICAL - Immediate Action'
    elif breach_prob >= 60:
        return 'HIGH - Monitor Closely'
    elif breach_prob >= 40:
        return 'MEDIUM - Track Progress'
    elif breach_prob >= 20:
        return 'LOW - Standard Process'
    else:
        return 'MINIMAL - On Track'

df_open['Risk_Tier'] = df_open['SLA_Breach_Probability'].apply(classify_risk)

print("\nRisk Tier Distribution:")
print(df_open['Risk_Tier'].value_counts())

In [ ]:
# 7. Calculate Response Time Factor
# If first response exists, calculate TTFR; if not, use ticket age
df_open['Hours_Since_First_Response'] = df_open.apply(
    lambda row: (datetime.now() - row['First_Response_At']).total_seconds() / 3600 
    if pd.notna(row['First_Response_At']) 
    else np.nan,
    axis=1
)

df_open['Has_Response'] = df_open['First_Response_At'].notna().astype(int)

print(f"\nTickets with First Response: {df_open['Has_Response'].sum()}")
print(f"Tickets Awaiting Response: {(1 - df_open['Has_Response']).sum()}")

In [ ]:
# 8. Prepare Risk Score Output
risk_scores_output = df_open[[
    'Ticket_Key',
    'Repo_Name',
    'Author_Name',
    'Created_At',
    'Ticket_Age_Hours',
    'Comment_Count',
    'Label_Weight',
    'Has_Response',
    'SLA_Breach_Probability',
    'Risk_Tier'
]].copy()

risk_scores_output['Calculated_At'] = datetime.now()

print("\nSample Risk Scores (Top 10 Highest Risk):")
print(risk_scores_output.nlargest(10, 'SLA_Breach_Probability')[['Ticket_Key', 'Repo_Name', 'Ticket_Age_Hours', 'Comment_Count', 'SLA_Breach_Probability', 'Risk_Tier']])

In [ ]:
# 9. Upload Risk Scores to SQL
risk_scores_output.to_sql(
    'Risk_Scores',
    con=engine,
    if_exists='replace',
    index=False
)

print("\n✅ Risk Scores uploaded to Risk_Scores table")
print(f"Total records: {len(risk_scores_output)}")
print(f"CRITICAL risk tickets: {(risk_scores_output['Risk_Tier'] == 'CRITICAL - Immediate Action').sum()}")
print(f"HIGH risk tickets: {(risk_scores_output['Risk_Tier'] == 'HIGH - Monitor Closely').sum()}")

In [ ]:
# 10. Summary Statistics
print("\n=== RISK SCORE SUMMARY ===")
print(f"\nTickets at Risk of SLA Breach (>60% prob): {(risk_scores_output['SLA_Breach_Probability'] >= 60).sum()}")
print(f"Average SLA Breach Probability: {risk_scores_output['SLA_Breach_Probability'].mean():.2f}%")
print(f"\nBy Risk Tier:")
print(risk_scores_output['Risk_Tier'].value_counts().to_string())

print(f"\n📊 High-Risk Repositories:")
high_risk_repos = risk_scores_output[risk_scores_output['SLA_Breach_Probability'] >= 60].groupby('Repo_Name').size().sort_values(ascending=False).head()
print(high_risk_repos.to_string())